#### 1. Import all libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import timedelta
from pathlib import Path
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

Now we need to import our dataset. For this, we need to mention the full part of the .csv file that we are going to use.

In [2]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "final_dataset.csv"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

print(OUTPUT_DIR)

C:\Users\amith\Documents\GitHub\resume-projects\Predictive Forecasting of Care Load & Placement Demand\outputs


Importing our dataset, sorting the dataset by date in ascending order

In [3]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df.sort_values(by=["Date"], inplace=True)
df.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-402.0,NaN,NaN,NaN,3,1,0
1,2023-01-13,33.0,53.0,34.0,6622.0,415.0,33.0,53.0,34.0,6566.0,...,NaN,NaN,NaN,-381.0,NaN,NaN,NaN,4,1,0
2,2023-01-14,33.0,52.0,35.0,6677.0,394.0,33.0,53.0,34.0,6622.0,...,NaN,NaN,NaN,-359.0,NaN,NaN,NaN,5,1,1
3,2023-01-15,33.0,52.0,36.0,6733.0,373.0,33.0,52.0,35.0,6677.0,...,NaN,NaN,NaN,-337.0,NaN,NaN,NaN,6,1,1
4,2023-01-16,33.0,51.0,36.0,6788.0,352.0,33.0,52.0,36.0,6733.0,...,NaN,NaN,NaN,-316.0,NaN,NaN,NaN,0,1,0
5,2023-01-17,32.0,51.0,36.0,6844.0,332.0,33.0,51.0,36.0,6788.0,...,NaN,NaN,NaN,-296.0,NaN,NaN,NaN,1,1,0
6,2023-01-18,32.0,51.0,37.0,6900.0,311.0,32.0,51.0,36.0,6844.0,...,NaN,NaN,NaN,-274.0,-337.857143,NaN,46.063104,2,1,0


Dropping all null values and resetting index

In [4]:
df_ml = df.dropna().reset_index(drop=True)
df_ml.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-26,21.0,28.0,34.0,7522.0,211.0,20.0,22.0,41.0,7538.0,...,37.714286,7035.071429,282.714286,-177.0,-173.571429,-245.000000,38.973740,3,1,0
1,2023-01-27,22.0,34.0,26.0,7505.0,242.0,21.0,28.0,34.0,7522.0,...,37.142857,7098.142857,270.357143,-216.0,-171.428571,-233.214286,35.552108,4,1,0
2,2023-01-28,22.0,39.0,18.0,7488.0,272.0,22.0,34.0,26.0,7505.0,...,35.928571,7156.071429,261.642857,-254.0,-177.714286,-225.714286,45.893770,5,1,1
3,2023-01-29,23.0,45.0,11.0,7472.0,303.0,22.0,39.0,18.0,7488.0,...,34.142857,7208.857143,256.642857,-292.0,-192.571429,-222.500000,63.308391,6,1,1
4,2023-01-30,34.0,54.0,29.0,7743.0,196.0,23.0,45.0,11.0,7472.0,...,33.642857,7277.071429,245.500000,-167.0,-196.142857,-211.857143,60.628455,0,1,0
5,2023-01-31,26.0,36.0,36.0,7803.0,158.0,34.0,54.0,29.0,7743.0,...,33.642857,7345.571429,233.071429,-122.0,-195.285714,-199.428571,61.783801,1,1,0
6,2023-02-01,25.0,32.0,27.0,7903.0,231.0,26.0,36.0,36.0,7803.0,...,32.928571,7417.214286,227.357143,-204.0,-204.571429,-194.428571,56.579906,2,2,0


In [5]:
print("Data loaded.")
print()
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Total days:", len(df))
print()
print("Date range after lag NaNs removed:", df_ml["Date"].min().date(), "to", df_ml["Date"].max().date())
print("Days usable for feature models (after lag NaNs removed):", len(df_ml))

Data loaded.

Date range: 2023-01-12 to 2025-12-21
Total days: 1075

Date range after lag NaNs removed: 2023-01-26 to 2025-12-21
Days usable for feature models (after lag NaNs removed): 1061


In [6]:
initial_day = df["Date"].iloc[0]
final_day = df["Date"].iloc[-1]
total_days = (final_day - initial_day).days

print(total_days)

1074


In [9]:
n_train = round(total_days * 0.67)
n_val = round(total_days * 0.25)


train_end = initial_day + timedelta(days = n_train - 1)
val_end = train_end + timedelta(days = n_val - 1)

print("Training dates range from : ",initial_day,"to",train_end)
print("Validation dates ranges from : ",train_end+timedelta(1),"to",train_end)

Training dates range from :  2023-01-12 00:00:00 to 2024-12-31 00:00:00
Validation dates ranges from :  2025-01-01 00:00:00 to 2024-12-31 00:00:00


In [13]:
train_df = df[df["Date"] <= train_end].copy()
val_df = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
test_df = df[df["Date"] > val_end].copy()

train_ml = df_ml[df_ml["Date"] <= train_end].copy()
test_ml = df_ml[df_ml["Date"] > val_end].copy()

In [14]:
print("\nSplit sizes:")
print("  Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))


Split sizes:
  Train: 720 | Val: 267 | Test: 88


In [26]:
def naive_forecast(series, horizon):
    """Repeat the last observed value for every future day."""
    last_value = float(series.iloc[-1])
    return np.full(horizon, last_value)

In [27]:
def moving_average_forecast(series, horizon, window=7):
    """Use the recent rolling mean as the constant forecast."""
    avg = float(series.tail(window).mean())
    return np.full(horizon, avg)

In [28]:
def arima_forecast(series, horizon, order=(2, 1, 2)):
    """Fit ARIMA and forecast horizon steps ahead."""
    try:
        model = ARIMA(series.astype(float), order=order)
        fitted = model.fit()
        pred = fitted.forecast(steps=horizon)
        return np.asarray(pred, dtype=float)
    except Exception as e:
        print("  ARIMA failed:", e)
        return naive_forecast(series, horizon)

In [34]:
FEATURE_COLS = [
    "Placed_In_CBP_lag_1", "CBP_Custody_lag_1", "CBP_To_HHS_lag_1",
    "HHS_Care_lag_1", "Discharged_From_HHS_lag_1",
    "Placed_In_CBP_lag_7", "CBP_Custody_lag_7", "CBP_To_HHS_lag_7",
    "HHS_Care_lag_7", "Discharged_From_HHS_lag_7",
    "Placed_In_CBP_lag_14", "CBP_Custody_lag_14", "CBP_To_HHS_lag_14",
    "HHS_Care_lag_14", "Discharged_From_HHS_lag_14",
    "HHS_Care_rolling_avg_7", "Discharged_From_HHS_rolling_avg_7",
    "HHS_Care_rolling_avg_14", "Discharged_From_HHS_rolling_avg_14",
    "Net_HHS_Flow", "Net_HHS_Flow_rolling_avg_7",
    "Day_Of_Week", "Month", "Is_Weekend"
]

In [35]:
def train_linear_model(train_frame, target):
    """
    Ordinary least-squares linear regression with numpy.
    Returns coefficient vector (including intercept).
    """
    X = train_frame[FEATURE_COLS].astype(float).values
    y = train_frame[target].astype(float).values
    # Add intercept column
    X_design = np.column_stack([np.ones(len(X)), X])
    # Solve least squares
    coeffs, residuals, rank, s = np.linalg.lstsq(X_design, y, rcond=None)
    return coeffs


In [36]:
def linear_predict(coeffs, frame):
    """Predict with the linear model coefficients."""
    X = frame[FEATURE_COLS].astype(float).values
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ coeffs

In [30]:
naive_forecast(train_df["HHS_Care"],7)

array([6430., 6430., 6430., 6430., 6430., 6430., 6430.])

In [31]:
moving_average_forecast(train_df["HHS_Care"],7)

array([6559.57142857, 6559.57142857, 6559.57142857, 6559.57142857,
       6559.57142857, 6559.57142857, 6559.57142857])

In [33]:
arima_forecast(train_df["HHS_Care"],10)

array([6471.64215182, 6498.18257942, 6491.98930537, 6460.56066979,
       6428.80028372, 6419.52555851, 6437.29208399, 6466.82957236,
       6485.84782431, 6481.68396183])

In [16]:
train_df

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-402.0,NaN,NaN,NaN,3,1,0
1,2023-01-13,33.0,53.0,34.0,6622.0,415.0,33.0,53.0,34.0,6566.0,...,NaN,NaN,NaN,-381.0,NaN,NaN,NaN,4,1,0
2,2023-01-14,33.0,52.0,35.0,6677.0,394.0,33.0,53.0,34.0,6622.0,...,NaN,NaN,NaN,-359.0,NaN,NaN,NaN,5,1,1
3,2023-01-15,33.0,52.0,36.0,6733.0,373.0,33.0,52.0,35.0,6677.0,...,NaN,NaN,NaN,-337.0,NaN,NaN,NaN,6,1,1
4,2023-01-16,33.0,51.0,36.0,6788.0,352.0,33.0,52.0,36.0,6733.0,...,NaN,NaN,NaN,-316.0,NaN,NaN,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,2024-12-27,89.0,138.0,146.0,6626.0,162.0,80.0,122.0,152.0,6707.0,...,204.642857,6570.000000,191.000000,-16.0,3.428571,13.642857,51.678033,4,12,0
716,2024-12-28,98.0,154.0,139.0,6546.0,189.0,89.0,138.0,146.0,6626.0,...,197.357143,6578.142857,189.214286,-50.0,1.000000,8.142857,54.021601,5,12,1
717,2024-12-29,107.0,170.0,133.0,6465.0,216.0,98.0,154.0,139.0,6546.0,...,191.214286,6582.214286,188.642857,-83.0,-6.285714,2.571429,62.055659,6,12,1
718,2024-12-30,99.0,156.0,139.0,6454.0,177.0,107.0,170.0,133.0,6465.0,...,187.642857,6582.214286,190.642857,-38.0,-4.000000,-3.000000,60.274373,0,12,0
